# Aula 02 — Perceptron e regra de aprendizagem

Este tutorial implementa o perceptron **somente com NumPy**. Vamos rastrear atualizações
exemplo a exemplo, verificar o limite de enganos em dados linearmente separáveis, medir a
dependência da ordem e demonstrar o que acontece em XOR, onde a separabilidade falha.

**Dependências mínimas:** Python 3.11, NumPy 1.26, Matplotlib 3.8 e nbformat 5.9.
Os dados são sintéticos, explícitos ou gerados por seed fixa; não há download nem segredo.


## Goal

Implementar a regra, testar seus contratos e separar três afirmações diferentes:

1. **correção da atualização:** um erro ou empate no limiar move o score na direção correta;
2. **convergência no treino:** há uma época sem erros quando os dados são separáveis;
3. **generalização:** desempenho em exemplos que não participaram das atualizações.

Rótulos usam $y\in\{-1,+1\}$. Predizemos $+1$ quando $w^Tx+b\ge0$ e $-1$ caso contrário.
Na predição, o empate $s=0$ é atribuído a $+1$; no treino, qualquer exemplo com
$y(w^Tx+b)\le0$ é tratado conservadoramente como violação e provoca a atualização
$w\leftarrow w+\eta yx$ e $b\leftarrow b+\eta y$.


## Setup

- exemplos em linhas: $X\in\mathbb{R}^{n\times d}$;
- peso $w\in\mathbb{R}^{d}$ e viés escalar $b$;
- treinamento online, uma linha por vez;
- `SEED = 20260909` para dados, split e embaralhamento;
- escala aprendida somente no treino;
- teste reservado nunca decide hiperparâmetro nem parada.


In [ ]:
from __future__ import annotations

import platform

import matplotlib
import matplotlib.pyplot as plt
import numpy as np

SEED = 20260909
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=6, suppress=True)

print({
    "python": platform.python_version(),
    "numpy": np.__version__,
    "matplotlib": matplotlib.__version__,
    "seed": SEED,
})


## Steps

### 1. Atualização manual em quatro exemplos

Começamos em $w=0$, $b=0$ e usamos $\eta=1$. O produto
$m_i=y_i(w^Tx_i+b)$ é a margem funcional: valores positivos indicam acerto; zero ou
negativo exige atualização. A tabela impressa registra o estado **antes** e **depois**.


In [ ]:
X_toy = np.array([
    [2.0, 1.0],
    [-1.0, -2.0],
    [1.0, 2.0],
    [-2.0, -1.0],
])
y_toy = np.array([1, -1, 1, -1])

w_toy = np.zeros(2)
b_toy = 0.0
trace = []
for step, (x_i, y_i) in enumerate(zip(X_toy, y_toy), start=1):
    score_before = float(w_toy @ x_i + b_toy)
    margin_before = float(y_i * score_before)
    updated = margin_before <= 0.0
    if updated:
        w_toy += y_i * x_i
        b_toy += float(y_i)
    trace.append({
        "passo": step,
        "y": int(y_i),
        "score_antes": round(score_before, 3),
        "margem_antes": round(margin_before, 3),
        "atualizou": updated,
        "w_depois": w_toy.copy(),
        "b_depois": b_toy,
    })

for row in trace:
    print(row)
assert np.array_equal(w_toy, np.array([2.0, 1.0]))
assert b_toy == 1.0


### 2. Implementação auditável

`fit_perceptron` conta atualizações, permite embaralhamento determinístico e interrompe apenas
quando uma época completa não produz atualização. O estado retornado inclui histórico,
número total de enganos e se a convergência foi observada dentro do orçamento.

O nome `mistakes` segue a literatura e o código didático, mas inclui o raro caso de margem
zero que a convenção de predição atribuiria a $+1$. Assim, a quantidade auditada é, com
precisão, o número de **violações que geraram atualização**.


In [ ]:
def decision_function(X: np.ndarray, w: np.ndarray, b: float) -> np.ndarray:
    X = np.asarray(X, dtype=float)
    assert X.ndim == 2 and w.shape == (X.shape[1],)
    return X @ w + b


def predict_perceptron(X: np.ndarray, w: np.ndarray, b: float) -> np.ndarray:
    scores = decision_function(X, w, b)
    return np.where(scores >= 0.0, 1, -1)


def fit_perceptron(
    X: np.ndarray,
    y: np.ndarray,
    *,
    learning_rate: float = 1.0,
    max_epochs: int = 100,
    shuffle: bool = True,
    seed: int = SEED,
):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=int)
    assert X.ndim == 2 and y.shape == (X.shape[0],)
    assert set(np.unique(y)).issubset({-1, 1})
    assert learning_rate > 0 and max_epochs >= 1

    local_rng = np.random.default_rng(seed)
    w = np.zeros(X.shape[1])
    b = 0.0
    mistakes_per_epoch = []
    total_mistakes = 0
    converged = False

    for _ in range(max_epochs):
        order = local_rng.permutation(len(X)) if shuffle else np.arange(len(X))
        mistakes = 0
        for idx in order:
            functional_margin = y[idx] * (w @ X[idx] + b)
            if functional_margin <= 0.0:
                w += learning_rate * y[idx] * X[idx]
                b += learning_rate * float(y[idx])
                mistakes += 1
                total_mistakes += 1
        mistakes_per_epoch.append(mistakes)
        if mistakes == 0:
            converged = True
            break

    return {
        "w": w,
        "b": b,
        "mistakes_per_epoch": np.asarray(mistakes_per_epoch),
        "total_mistakes": total_mistakes,
        "epochs": len(mistakes_per_epoch),
        "converged": converged,
    }


### 3. Sanidade local da regra

Para um exemplo classificado incorretamente, a mudança do próprio score é

$$\Delta s_i=\eta y_i(\|x_i\|^2+1),$$

onde o termo 1 vem do viés. Portanto $y_i\Delta s_i>0$: a margem funcional daquele
exemplo aumenta estritamente após a atualização.


In [ ]:
x_check = np.array([0.5, -2.0])
y_check = -1
w_check = np.array([1.0, 0.0])
b_check = 1.0
eta_check = 0.25

score_before = float(w_check @ x_check + b_check)
margin_before = y_check * score_before
assert margin_before <= 0.0

w_after = w_check + eta_check * y_check * x_check
b_after = b_check + eta_check * y_check
score_after = float(w_after @ x_check + b_after)
margin_after = y_check * score_after
expected_gain = eta_check * (np.dot(x_check, x_check) + 1.0)

assert np.isclose(margin_after - margin_before, expected_gain)
assert margin_after > margin_before
print({
    "margem_antes": margin_before,
    "margem_depois": margin_after,
    "ganho": margin_after - margin_before,
    "ganho_previsto": expected_gain,
})


### 4. População sintética separável e teste reservado

Geramos pontos uniformes e removemos uma faixa em torno da fronteira verdadeira. Isso
cria margem positiva por construção. Separamos 25% para teste **antes** de estimar média e
desvio; a padronização é ajustada apenas no treino.


In [ ]:
n_candidates = 2000
X_candidates = rng.uniform(-3.0, 3.0, size=(n_candidates, 2))
w_true_raw = np.array([1.25, -0.8])
b_true_raw = 0.3
raw_scores = X_candidates @ w_true_raw + b_true_raw
keep = np.abs(raw_scores) >= 0.45
X_population = X_candidates[keep][:480]
y_population = np.where(
    X_population @ w_true_raw + b_true_raw >= 0.0, 1, -1
)

indices = rng.permutation(len(X_population))
n_test = len(X_population) // 4
test_idx = indices[:n_test]
train_idx = indices[n_test:]
assert set(test_idx).isdisjoint(set(train_idx))

X_train_raw, X_test_raw = X_population[train_idx], X_population[test_idx]
y_train, y_test = y_population[train_idx], y_population[test_idx]
train_mean = X_train_raw.mean(axis=0)
train_std = X_train_raw.std(axis=0, ddof=0)
X_train = (X_train_raw - train_mean) / train_std
X_test = (X_test_raw - train_mean) / train_std

assert X_train.shape == (360, 2) and X_test.shape == (120, 2)
assert np.all(train_std > 0)
assert np.allclose(X_train.mean(axis=0), 0.0, atol=1e-14)
print({
    "treino": X_train.shape,
    "teste_reservado": X_test.shape,
    "classes_treino": dict(zip(*np.unique(y_train, return_counts=True))),
    "media_treino": X_train.mean(axis=0),
    "media_teste": X_test.mean(axis=0),
})


### 5. Convergência observada e limite de enganos

No espaço padronizado, a fronteira verdadeira é reescrita exatamente. Acrescentamos 1 a
cada entrada para incorporar o viés: $\tilde{x}=[x;1]$ e $\tilde{w}=[w;b]$.
Normalizamos o separador verdadeiro $u$ para norma 1 e calculamos

$$R=\max_i\|\tilde{x}_i\|,\qquad
\gamma=\min_i y_i u^T\tilde{x}_i.$$

Se $\gamma>0$, o limite clássico é $M\le(R/\gamma)^2$ enganos. Ele é um teto, não uma
previsão do número exato.


In [ ]:
separable_run = fit_perceptron(X_train, y_train, max_epochs=200, seed=SEED)
w_sep, b_sep = separable_run["w"], separable_run["b"]
train_pred = predict_perceptron(X_train, w_sep, b_sep)
test_pred = predict_perceptron(X_test, w_sep, b_sep)
train_accuracy = float(np.mean(train_pred == y_train))
test_accuracy = float(np.mean(test_pred == y_test))

# score_raw = w_raw @ (mean + std*x_std) + b_raw
w_true_std = w_true_raw * train_std
b_true_std = float(w_true_raw @ train_mean + b_true_raw)
separator_aug = np.r_[w_true_std, b_true_std]
unit_separator = separator_aug / np.linalg.norm(separator_aug)
X_train_aug = np.c_[X_train, np.ones(len(X_train))]
R = float(np.max(np.linalg.norm(X_train_aug, axis=1)))
gamma = float(np.min(y_train * (X_train_aug @ unit_separator)))
mistake_bound = float((R / gamma) ** 2)

assert gamma > 0.0
assert separable_run["converged"]
assert train_accuracy == 1.0
assert separable_run["total_mistakes"] <= mistake_bound + 1e-12
print({
    "convergiu": separable_run["converged"],
    "epocas": separable_run["epochs"],
    "enganos": separable_run["total_mistakes"],
    "R": round(R, 6),
    "gamma": round(gamma, 6),
    "limite_R2_gamma2": round(mistake_bound, 3),
    "acuracia_treino": train_accuracy,
    "acuracia_teste": test_accuracy,
})


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(np.arange(1, separable_run["epochs"] + 1),
             separable_run["mistakes_per_epoch"], marker="o")
axes[0].set(xlabel="época", ylabel="atualizações",
            title="Dados separáveis: enganos até convergir")
axes[0].grid(alpha=0.3)

axes[1].scatter(X_train[:, 0], X_train[:, 1], c=y_train,
                cmap="coolwarm", alpha=0.45, s=20, label="treino")
axes[1].scatter(X_test[:, 0], X_test[:, 1], c=y_test,
                cmap="coolwarm", edgecolor="black", s=35, marker="s", label="teste")
x_line = np.linspace(X_train[:, 0].min(), X_train[:, 0].max(), 200)
y_line = -(w_sep[0] * x_line + b_sep) / w_sep[1]
axes[1].plot(x_line, y_line, "k--", label="fronteira aprendida")
axes[1].set(xlabel="$x_1$ padronizado", ylabel="$x_2$ padronizado",
            title="Separação linear e teste reservado")
axes[1].legend(loc="best")
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()


### 6. A solução depende da ordem

O teorema garante que algum separador será encontrado sob as hipóteses; não garante pesos
únicos. Repetimos o treino com 30 ordens. Para comparar fronteiras, normalizamos o vetor
ampliado $[w;b]$ e fixamos sua orientação.


In [ ]:
order_runs = []
unit_solutions = []
for seed in range(30):
    run = fit_perceptron(X_train, y_train, max_epochs=200, seed=seed)
    vector = np.r_[run["w"], run["b"]]
    vector = vector / np.linalg.norm(vector)
    if vector @ unit_separator < 0:
        vector = -vector
    unit_solutions.append(vector)
    order_runs.append((run["total_mistakes"], run["epochs"], run["converged"]))

unit_solutions = np.asarray(unit_solutions)
mistakes_by_order = np.array([row[0] for row in order_runs])
epochs_by_order = np.array([row[1] for row in order_runs])
pairwise_cosine = unit_solutions @ unit_solutions.T
upper = pairwise_cosine[np.triu_indices_from(pairwise_cosine, k=1)]

assert all(row[2] for row in order_runs)
assert np.unique(np.round(unit_solutions, 8), axis=0).shape[0] > 1
print({
    "enganos_min_mediana_max": (
        int(mistakes_by_order.min()),
        float(np.median(mistakes_by_order)),
        int(mistakes_by_order.max()),
    ),
    "epocas_min_max": (int(epochs_by_order.min()), int(epochs_by_order.max())),
    "cosseno_minimo_entre_solucoes": round(float(upper.min()), 6),
    "todas_convergiram": True,
})


### 7. Score não é probabilidade

Multiplicar $w$ e $b$ por uma constante positiva preserva toda classe, mas altera a
magnitude dos scores. Portanto, `score = 12` não significa “probabilidade maior” que
`score = 3` em sentido calibrado.


In [ ]:
scores_original = decision_function(X_test, w_sep, b_sep)
scale_factor = 50.0
scores_scaled = decision_function(X_test, scale_factor * w_sep, scale_factor * b_sep)
pred_scaled = predict_perceptron(X_test, scale_factor * w_sep, scale_factor * b_sep)

assert np.array_equal(pred_scaled, test_pred)
assert np.allclose(scores_scaled, scale_factor * scores_original)
print({
    "concordancia_de_classes": float(np.mean(pred_scaled == test_pred)),
    "mediana_abs_score_original": round(float(np.median(np.abs(scores_original))), 6),
    "mediana_abs_score_escalado": round(float(np.median(np.abs(scores_scaled))), 6),
})


### 8. Não separabilidade: XOR e orçamento explícito

Em XOR, nenhuma reta classifica os quatro pontos. O perceptron padrão não possui um mínimo
suave para perseguir: continua atualizando enquanto encontra erros. Um orçamento de épocas
é obrigatório. O algoritmo *pocket* guarda o estado com menor número de erros observado;
ele é uma mitigação prática, não restaura a garantia de convergência.


In [ ]:
X_xor = np.array([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
y_xor = np.array([-1, 1, 1, -1])
xor_run = fit_perceptron(
    X_xor, y_xor, max_epochs=80, shuffle=False, seed=SEED
)
xor_pred = predict_perceptron(X_xor, xor_run["w"], xor_run["b"])
xor_final_accuracy = float(np.mean(xor_pred == y_xor))

assert not xor_run["converged"]
assert np.all(xor_run["mistakes_per_epoch"] > 0)
assert xor_final_accuracy < 1.0
print({
    "convergiu": xor_run["converged"],
    "epocas_executadas": xor_run["epochs"],
    "enganos_totais": xor_run["total_mistakes"],
    "enganos_ultimas_5_epocas": xor_run["mistakes_per_epoch"][-5:],
    "acuracia_final": xor_final_accuracy,
})


In [ ]:
def fit_pocket(X, y, *, max_epochs=80, seed=SEED):
    local_rng = np.random.default_rng(seed)
    w = np.zeros(X.shape[1])
    b = 0.0
    best_w, best_b = w.copy(), b
    best_errors = int(np.sum(predict_perceptron(X, w, b) != y))
    history = []

    for _ in range(max_epochs):
        for idx in local_rng.permutation(len(X)):
            if y[idx] * (w @ X[idx] + b) <= 0.0:
                w += y[idx] * X[idx]
                b += float(y[idx])
                errors = int(np.sum(predict_perceptron(X, w, b) != y))
                if errors < best_errors:
                    best_errors = errors
                    best_w, best_b = w.copy(), b
        history.append(best_errors)
    return best_w, best_b, best_errors, np.asarray(history)


w_pocket, b_pocket, pocket_errors, pocket_history = fit_pocket(X_xor, y_xor)
pocket_accuracy = float(np.mean(predict_perceptron(X_xor, w_pocket, b_pocket) == y_xor))
assert pocket_errors == 1
assert pocket_accuracy == 0.75

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(np.arange(1, xor_run["epochs"] + 1), xor_run["mistakes_per_epoch"],
        label="perceptron: atualizações/época")
ax.plot(np.arange(1, len(pocket_history) + 1), pocket_history,
        label="pocket: melhor nº de erros")
ax.set(xlabel="época", ylabel="contagem", title="XOR: ausência de convergência")
ax.legend()
ax.grid(alpha=0.3)
plt.show()
print({"erros_no_pocket": pocket_errors, "acuracia_pocket": pocket_accuracy})


## Checks

As verificações abaixo cobrem atualização local, isolamento do teste, separabilidade,
limite de enganos, dependência da ordem, invariância da classe por escala e falha esperada
em XOR. Elas não transformam o perceptron em estimador probabilístico nem provam validade
fora da população sintética.


In [ ]:
checks = {
    "rotulos_validos": set(np.unique(y_train)) == {-1, 1},
    "split_disjunto": set(train_idx).isdisjoint(set(test_idx)),
    "escala_so_do_treino": np.allclose(X_train.mean(axis=0), 0.0, atol=1e-14),
    "atualizacao_aumenta_margem": margin_after > margin_before,
    "gamma_positivo": gamma > 0.0,
    "convergencia_observada": separable_run["converged"],
    "treino_perfeito": train_accuracy == 1.0,
    "enganos_dentro_do_limite": separable_run["total_mistakes"] <= mistake_bound,
    "ordem_muda_solucao": np.unique(np.round(unit_solutions, 8), axis=0).shape[0] > 1,
    "escala_preserva_classes": np.array_equal(pred_scaled, test_pred),
    "xor_nao_converge": not xor_run["converged"],
    "pocket_guarda_melhor": pocket_errors == 1,
}
assert all(checks.values())
print(checks)
print(f"{sum(checks.values())}/{len(checks)} contratos satisfeitos")


## Next Steps

- O perceptron é um classificador linear online, não uma probabilidade.
- A regra corrige apenas exemplos com margem funcional não positiva.
- Separabilidade com margem positiva implica um número finito de enganos.
- A garantia não escolhe solução única e não vale para XOR ou rótulos contraditórios.
- *Pocket* conserva o melhor estado observado, mas exige avaliação e critério externo.

Na **Aula 03**, organizaremos vários neurônios em camadas densas e fixaremos convenções de
shape para uma rede de duas camadas. O backpropagation completo será construído depois,
operação por operação, conforme a grade canônica.
